In [2]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/lymphoma_classifier"

Mounted at /content/drive


In [3]:
# Cell 2 — Reinstall dependencies (Colab doesn't remember installs across notebooks)
!pip install openslide-python -q
!apt-get install -y openslide-tools -q

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 42 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 0s (686 kB/s)
Selecting previously unselected package libopenslide0.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libopenslide0_3.4.1+dfsg-5build1_amd64.deb ...
Unpacking libopenslide0 (3.4.1+dfsg-5build1) ...
Selecting previously unselected package openslide-tools.
Preparing to u

In [4]:
# Cell 3 — Confirm patches are accessible
import os
for label in ["burkitt", "dlbcl"]:
    folder = os.path.join(BASE_DIR, "patches", label)
    count = len([f for f in os.listdir(folder) if f.endswith(".png")])
    print(f"{label}: {count} patches")

burkitt: 3278 patches
dlbcl: 4217 patches


Cell 1 — Imports and config (paste and run)


Cell 2 — PatchDataset class (paste and run)


Cell 3 — Transforms and splits (paste and run, share the output — it will show train/val/test counts)


Cell 4 — Model (paste and run, should print Linear(in_features=512, out_features=2, bias=True))


Cell 5 — Training loop (paste and run — this is the big one, will take 20–40 minutes)


Cell 6 — Evaluation (run after training finishes)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from tqdm import tqdm

# Config
PATCH_DIR = f"{BASE_DIR}/patches"
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
class PatchDataset(Dataset):
    def __init__(self, samples, transform=None):
        """
        samples: list of (patch_path, label) tuples
        transform: image augmentations/preprocessing
        """
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
# Transforms — training gets augmentation, val/test get just normalization
# ImageNet mean/std because ResNet was pretrained on ImageNet
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Build full sample list [(path, label), ...]
label_map = {"burkitt": 0, "dlbcl": 1}
all_samples = []
for label_name, label_idx in label_map.items():
    folder = os.path.join(PATCH_DIR, label_name)
    for fname in os.listdir(folder):
        if fname.endswith(".png"):
            all_samples.append((os.path.join(folder, fname), label_idx))

# Shuffle and split
np.random.seed(42)
np.random.shuffle(all_samples)

n = len(all_samples)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
n_test = n - n_train - n_val

train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:n_train + n_val]
test_samples = all_samples[n_train + n_val:]

train_dataset = PatchDataset(train_samples, transform=train_transform)
val_dataset = PatchDataset(val_samples, transform=val_test_transform)
test_dataset = PatchDataset(test_samples, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Total patches: {n}")
print(f"  Train: {len(train_dataset)}")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")

Total patches: 7495
  Train: 5246
  Val:   1124
  Test:  1125


In [ ]:
def build_model():
    # Load ResNet18 pretrained on ImageNet
    model = models.resnet18(weights="IMAGENET1K_V1")

    # Replace the final layer for binary classification
    # ResNet18's final layer outputs 1000 classes — we need just 2
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 2)

    return model.to(DEVICE)

model = build_model()
print(model.fc)  # confirm the final layer is now Linear(512 → 2)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

# Training loop
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f"{BASE_DIR}/best_model.pth")
        print(f"  ✓ New best model saved")

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load(f"{BASE_DIR}/best_model.pth"))
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion matrix
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Burkitt", "DLBCL"],
            yticklabels=["Burkitt", "DLBCL"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — Test Set")
plt.show()

print(classification_report(all_labels, all_preds,
                            target_names=["Burkitt", "DLBCL"]))

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"], label="Val")
ax1.set_title("Loss")
ax1.legend()
ax2.plot(history["train_acc"], label="Train")
ax2.plot(history["val_acc"], label="Val")
ax2.set_title("Accuracy")
ax2.legend()
plt.show()